# 04 — デモ: Session 3b 不整地 perlin（本番デモ）

**目的:** 連続起伏 terrain でコンサル本番デモ。Session 3a の知見 + さらに保守的チューニング。


## 4セッションの違い（必読）

| | **S1 本Notebook** | S2 tune | S3a boxes | **S3b perlin ← 今ここ** |
|---|-------------------|---------|-----------|------------|
| **scene** | **flat（平坦）** | flat | **random_boxes（箱）** | **perlin（連続起伏）** |
| **足場最適化** | **OFF** | OFF | ON | ON |
| **主な目的** | 最小構成で動作確認 | μ / 歩調チューニング | 段差・離散障害 | 連続起伏 |
| **デモGIFで見る点** | 平坦＋標準trot | 平坦＋**速いtrot** | **箱が見える** | **うねり地形** |
| **GIF** | demo_s01_flat | demo_s02_tune | demo_s03_boxes | demo_s03_perlin |

> **S1 と S2 は地形とも平坦**です。GIFの違いは **歩調（S2は step_freq=1.75 Hz の速い trot）** と **Notebook内の実験内容** です。  
> **S3a/S3b は約9秒走行**して箱・起伏地形に入るようキャプチャしています（旧GIFは短すぎて全部平坦に見えていました）。


### このセッション固有のポイント

- **地形:** `scene=perlin` — **height field** による連続的なうねり（箱のような段差ではない）
- **足場最適化:** ON（S3a と同様）
- **チューニング:** step_freq=1.15 / duty=0.75 / mu=0.45 と **S3a より保守的**
- **デモGIF:** 低めカメラ＋約9秒走行で **地面の起伏** が見える。左上 `Session 3b | scene=perlin`
- **S3a との違い:** boxes=離散障害、perlin=連続起伏（難易度・見た目・調整方針が異なる）

![Session 3b demo](../assets/demo_s03_perlin.gif)


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 1 — プリセット確認

In [ ]:
import yaml
print(yaml.dump(load_preset_yaml("session03_rough_perlin"), allow_unicode=True))


| 項目 | 値 | 意図 |
|------|-----|------|
| scene | perlin | 連続起伏 |
| step_freq | 1.15 | 低め |
| duty_factor | 0.75 | 支持長め |
| mu | 0.45 | 安全側 |


## Step 2 — headless 5秒検証

In [ ]:
apply_preset("session03_rough_perlin")
m = run_flat_sim(seconds=5.0, scene="perlin")
print(m)
fig = compare_runs([("perlin preset", m)])
plt.show()


## Step 3 — ❌ vs ✅ mu の安全側調整

perlin では **積極的な mu は転倒要因** になりやすい。


In [ ]:
apply_preset("session03_rough_perlin")
runs = []
for mu, label in [(0.55, "mu=0.55 aggressive"), (0.45, "mu=0.45 baseline"), (0.35, "mu=0.35 conservative")]:
    r = run_flat_sim(seconds=4.0, scene="perlin", mu=mu)
    runs.append((label, r))
fig = compare_runs(runs)
plt.show()


## Step 4 — boxes vs perlin 比較（難易度の違い）

Session 3a の結果と並べて説明すると効果的。


In [ ]:
apply_preset("session03_rough_boxes")
boxes = run_flat_sim(seconds=4.0, scene="random_boxes")
apply_preset("session03_rough_perlin")
perlin = run_flat_sim(seconds=4.0, scene="perlin")
fig = compare_runs([("random_boxes", boxes), ("perlin", perlin)])
plt.suptitle("terrain difficulty comparison", y=1.02)
plt.show()


## Step 5 — 本番デモ脚本

1. Session 1 GIF → 3層説明（平坦・足場opt OFF）  
2. Session 2 Notebook → μ / 歩調（平坦・パラメータ実験）  
3. Session 3 perlin GIF → 足場 opt + 連続起伏  

映像: [`../assets/demo_s03_perlin.gif`](../assets/demo_s03_perlin.gif)

---

## Step 6 — ワークショップ完了チェック

- [ ] Session 1–3 すべて再現可能  
- [ ] 失敗時のトリアージを **TUNING_GUIDE** から選べる  
- [ ] 4セッションの **地形・足場opt・目的** の違いを説明できる  
- [ ] お客様向け1文: 「MPCが **どこに足を置き、どれだけ蹴るか** を計画」  

---

## 次へのステップ

- 実機: muse + `ros2/run_controller.py`  
- 理論復習: [00_theory_grf_mpc_wbc.ipynb](./00_theory_grf_mpc_wbc.ipynb)  
- 統合資料: [WORKSHOP.md](../WORKSHOP.md)
